# Lesson 5 — Entanglement and Bell States

**Quantum Computing with Qiskit**

This notebook accompanies Lesson 5. Work through the cells in order. Every cell is meant to be run; modify the code freely and re-run to experiment.

**Topics covered:**
- How H and CNOT create entanglement
- The four Bell states and the Bell measurement
- Measuring correlations: entangled vs separable
- GHZ state and W state circuits
- GHZ vs W: entanglement under qubit loss
- CHSH inequality: quantum violation of the classical bound

In [ ]:
!pip install qiskit qiskit-aer qiskit-ibm-runtime pylatexenc --quiet

In [ ]:
import qiskit
import qiskit_aer
import qiskit_ibm_runtime

print("Qiskit:             ", qiskit.__version__)
print("Qiskit Aer:         ", qiskit_aer.__version__)
print("Qiskit IBM Runtime: ", qiskit_ibm_runtime.__version__)

## 1. How H and CNOT Create Entanglement

A Hadamard puts the control qubit in superposition; CNOT then correlates both qubits:

$$\frac{1}{\sqrt{2}}(|0\rangle + |1\rangle) \otimes |0\rangle \xrightarrow{CX} \frac{1}{\sqrt{2}}(|00\rangle + |11\rangle)$$

The result cannot be written as $|\psi_0\rangle \otimes |\psi_1\rangle$ — the qubits are entangled.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)

sv = Statevector.from_label('00').evolve(qc)
print(sv)
print("Probabilities:", sv.probabilities_dict())

qc.draw('mpl')

## 2. The Four Bell States

$$|\Phi^+\rangle = \frac{1}{\sqrt{2}}(|00\rangle + |11\rangle), \qquad |\Phi^-\rangle = \frac{1}{\sqrt{2}}(|00\rangle - |11\rangle)$$
$$|\Psi^+\rangle = \frac{1}{\sqrt{2}}(|01\rangle + |10\rangle), \qquad |\Psi^-\rangle = \frac{1}{\sqrt{2}}(|01\rangle - |10\rangle)$$

All four share the same circuit structure. Optional X gates select which state is prepared.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

def make_bell(x0=False, x1=False):
    """Prepare a Bell state. x0 and x1 select which of the four states."""
    qc = QuantumCircuit(2)
    if x0:
        qc.x(0)
    if x1:
        qc.x(1)
    qc.h(0)
    qc.cx(0, 1)
    return qc

configs = [
    (False, False, '|Φ+⟩'),
    (True,  False, '|Φ-⟩'),
    (False, True,  '|Ψ+⟩'),
    (True,  True,  '|Ψ-⟩'),
]

for x0, x1, label in configs:
    qc = make_bell(x0, x1)
    sv = Statevector.from_label('00').evolve(qc)
    print(f"{label}: {sv.data.round(3)}")

In [ ]:
# Draw all four circuits
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 4, figsize=(16, 3))
for ax, (x0, x1, label) in zip(axes, configs):
    make_bell(x0, x1).draw('mpl', ax=ax)
    ax.set_title(label)
plt.tight_layout()
plt.show()

### Bell measurement

The inverse Bell circuit (CNOT then H) decodes each Bell state to a unique computational basis outcome.

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

sim = AerSimulator()

for x0, x1, label in configs:
    qc = QuantumCircuit(2, 2)
    if x0:
        qc.x(0)
    if x1:
        qc.x(1)
    qc.h(0)
    qc.cx(0, 1)
    # Decode: inverse Bell circuit
    qc.cx(0, 1)
    qc.h(0)
    qc.measure([0, 1], [0, 1])

    counts = sim.run(transpile(qc, sim), shots=1024).result().get_counts()
    print(f"{label}: {counts}")

## 3. Measuring Correlations

The entangled state shows perfectly correlated outcomes. A separable state (H on both qubits independently) shows all four outcomes with equal probability.

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

sim = AerSimulator()

# Entangled: |Φ+⟩
qc_bell = QuantumCircuit(2, 2)
qc_bell.h(0)
qc_bell.cx(0, 1)
qc_bell.measure([0, 1], [0, 1])
counts_bell = sim.run(transpile(qc_bell, sim), shots=4096).result().get_counts()

# Separable: H on both qubits independently
qc_sep = QuantumCircuit(2, 2)
qc_sep.h(0)
qc_sep.h(1)
qc_sep.measure([0, 1], [0, 1])
counts_sep = sim.run(transpile(qc_sep, sim), shots=4096).result().get_counts()

plot_histogram(
    [counts_bell, counts_sep],
    legend=['|Φ+⟩ (entangled)', '(H⊗H)|00⟩ (separable)'],
    title='Entangled vs separable measurement statistics'
)

## 4. GHZ and W States

Three qubits admit two inequivalent entanglement classes. No local operations can convert one into the other.

### GHZ state

$$|\text{GHZ}\rangle = \frac{1}{\sqrt{2}}(|000\rangle + |111\rangle)$$

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

qc_ghz = QuantumCircuit(3)
qc_ghz.h(0)
qc_ghz.cx(0, 1)
qc_ghz.cx(1, 2)

sv_ghz = Statevector.from_label('000').evolve(qc_ghz)
print("GHZ statevector:", sv_ghz.data.round(3))
print("GHZ probabilities:", sv_ghz.probabilities_dict())

qc_ghz.draw('mpl')

### W state

$$|W\rangle = \frac{1}{\sqrt{3}}(|001\rangle + |010\rangle + |100\rangle)$$

The circuit uses $R_y(2\arcsin(1/\sqrt{3}))$, a controlled-H, and a Toffoli gate.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

def make_W_state():
    theta = 2 * np.arcsin(1 / np.sqrt(3))   # sin(theta/2) = 1/√3

    qc = QuantumCircuit(3)

    # Step 1: put amplitude 1/√3 on |q0=1⟩
    qc.ry(theta, 0)

    # Step 2: if original q0=0, spread to q1 using controlled-H
    qc.x(0)          # temporarily invert q0
    qc.ch(0, 1)      # H on q1 when q0=|1⟩ (i.e. original q0=|0⟩)

    # Step 3: if original q0=0 and q1=0, flip q2
    qc.x(1)
    qc.ccx(0, 1, 2)  # Toffoli: flip q2 when q0=1 and q1=1 (original: q0=0, q1=0)
    qc.x(0)          # restore q0
    qc.x(1)          # restore q1
    return qc

qc_w = make_W_state()

sv_w = Statevector.from_label('000').evolve(qc_w)
print("W statevector:", sv_w.data.round(3))
print("W probabilities:", sv_w.probabilities_dict())

qc_w.draw('mpl')

Amplitude $1/\sqrt{3} \approx 0.577$ at exactly three indices: `'001'`, `'010'`, `'100'`. Each outcome has probability 1/3.

### GHZ vs W: comparing measurement statistics

In [ ]:
from qiskit import transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

sim = AerSimulator()

qc_ghz_m = qc_ghz.copy()
qc_ghz_m.measure_all()
counts_ghz = sim.run(transpile(qc_ghz_m, sim), shots=4096).result().get_counts()

qc_w_m = qc_w.copy()
qc_w_m.measure_all()
counts_w = sim.run(transpile(qc_w_m, sim), shots=4096).result().get_counts()

plot_histogram(
    [counts_ghz, counts_w],
    legend=['GHZ', 'W'],
    title='GHZ vs W state measurement statistics'
)

### GHZ vs W: entanglement under qubit loss

Measure and discard qubit 2, then observe what happens to qubits 0 and 1.

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

sim = AerSimulator()

# GHZ: measure qubit 2 then measure qubits 0 and 1
qc_ghz_partial = QuantumCircuit(3, 3)
qc_ghz_partial.h(0)
qc_ghz_partial.cx(0, 1)
qc_ghz_partial.cx(1, 2)
qc_ghz_partial.measure(2, 2)           # discard qubit 2
qc_ghz_partial.measure([0, 1], [0, 1])

counts_ghz_p = sim.run(transpile(qc_ghz_partial, sim), shots=4096).result().get_counts()

# W: same structure
qc_w_partial = QuantumCircuit(3, 3)
theta = 2 * np.arcsin(1 / np.sqrt(3))
qc_w_partial.ry(theta, 0)
qc_w_partial.x(0)
qc_w_partial.ch(0, 1)
qc_w_partial.x(1)
qc_w_partial.ccx(0, 1, 2)
qc_w_partial.x(0)
qc_w_partial.x(1)
qc_w_partial.measure(2, 2)             # discard qubit 2
qc_w_partial.measure([0, 1], [0, 1])

counts_w_p = sim.run(transpile(qc_w_partial, sim), shots=4096).result().get_counts()

print("GHZ after discarding qubit 2:")
print({k: v for k, v in sorted(counts_ghz_p.items())})
print()
print("W after discarding qubit 2:")
print({k: v for k, v in sorted(counts_w_p.items())})

For GHZ: qubits 0 and 1 always agree (both 0 or both 1). Discarding qubit 2 leaves a classical mixture with no entanglement between the remaining pair.

For W: when qubit 2 measured 1, qubits 0 and 1 are both 0. When qubit 2 measured 0, outcomes `'010'` and `'001'` appear with equal probability — the signature of entanglement between the surviving pair.

## 5. CHSH Inequality

The CHSH inequality bounds classical correlations:

$$|S| = |E(a,b) + E(a,b') + E(a',b) - E(a',b')| \leq 2 \quad (\text{classical})$$

For $|\Phi^+\rangle$ with measurement rotation $R_y(2\theta)$, the correlation is $E(\alpha,\beta) = \cos(2(\alpha-\beta))$.
With optimal angles $a=0$, $a'=\pi/4$, $b=\pi/8$, $b'=-\pi/8$, the quantum value is $S = 2\sqrt{2} \approx 2.828$.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

sim = AerSimulator()

def measure_correlation(angle_a, angle_b, shots=8192):
    """
    Measure the CHSH correlation E(a, b) for |Φ+⟩.
    Applies Ry(2*angle) to each qubit before measuring in Z.
    Returns E = P(same outcome) - P(different outcome).
    """
    qc = QuantumCircuit(2, 2)
    qc.h(0)
    qc.cx(0, 1)
    qc.ry(2 * angle_a, 0)   # Alice's measurement rotation
    qc.ry(2 * angle_b, 1)   # Bob's measurement rotation
    qc.measure([0, 1], [0, 1])

    counts = sim.run(transpile(qc, sim), shots=shots).result().get_counts()
    n00 = counts.get('00', 0)
    n01 = counts.get('01', 0)
    n10 = counts.get('10', 0)
    n11 = counts.get('11', 0)

    return (n00 + n11 - n01 - n10) / shots

# Optimal CHSH angles
a   = 0
a_p = np.pi / 4
b   = np.pi / 8
b_p = -np.pi / 8

Eab    = measure_correlation(a,   b)
Eab_p  = measure_correlation(a,   b_p)
Ea_pb  = measure_correlation(a_p, b)
Ea_pbp = measure_correlation(a_p, b_p)

S = Eab + Eab_p + Ea_pb - Ea_pbp

print(f"E(a,  b ) = {Eab:.3f}   (expected:  {np.cos(2*(a   - b  )):.3f})")
print(f"E(a,  b') = {Eab_p:.3f}   (expected:  {np.cos(2*(a   - b_p)):.3f})")
print(f"E(a', b ) = {Ea_pb:.3f}   (expected:  {np.cos(2*(a_p - b  )):.3f})")
print(f"E(a', b') = {Ea_pbp:.3f}   (expected:  {np.cos(2*(a_p - b_p)):.3f})")
print()
print(f"CHSH value S = {S:.3f}")
print(f"Classical bound: |S| ≤ 2")
print(f"Quantum maximum: 2√2 ≈ {2 * np.sqrt(2):.3f}")
print(f"Violation: {'YES' if abs(S) > 2 else 'NO'}")

### Scanning the CHSH value across angles

In [ ]:
import matplotlib.pyplot as plt

a_primes = np.linspace(0, np.pi / 2, 20)
S_values = []
S_theory = []

for a_p_scan in a_primes:
    Eab_s    = measure_correlation(a,        b,   shots=2048)
    Eab_ps   = measure_correlation(a,        b_p, shots=2048)
    Ea_pbs   = measure_correlation(a_p_scan, b,   shots=2048)
    Ea_pbps  = measure_correlation(a_p_scan, b_p, shots=2048)
    S_values.append(Eab_s + Eab_ps + Ea_pbs - Ea_pbps)
    S_theory.append(
        np.cos(2*(a - b)) + np.cos(2*(a - b_p)) +
        np.cos(2*(a_p_scan - b)) - np.cos(2*(a_p_scan - b_p))
    )

plt.figure(figsize=(8, 4))
plt.axhline( 2, color='gray', linestyle='--', label='Classical bound +2')
plt.axhline(-2, color='gray', linestyle='--', label='Classical bound -2')
plt.plot(a_primes / np.pi, S_theory, label='Theoretical S')
plt.scatter(a_primes / np.pi, S_values, label='Measured S', zorder=5)
plt.xlabel("$a' / \\pi$")
plt.ylabel('CHSH value S')
plt.title("CHSH Parameter vs Alice's Second Angle $a'$")
plt.legend()
plt.tight_layout()
plt.show()

## 6. Exercises

### Exercise 1: Correlations in all four Bell states

For each of the four Bell states, measure 4096 shots in the Z basis and plot all four histograms side by side.

1. Which Bell states show same-bit correlations (`'00'` and `'11'` only)?
2. Which show opposite-bit correlations (`'01'` and `'10'` only)?
3. Can you distinguish $|\Phi^+\rangle$ from $|\Phi^-\rangle$ in the Z basis? What basis would separate them?

In [ ]:
# Exercise 1: your solution here
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

sim = AerSimulator()
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, (x0, x1, label) in zip(axes, configs):
    qc = QuantumCircuit(2, 2)
    # TODO: prepare Bell state and measure
    pass

plt.tight_layout()
plt.show()

### Exercise 2: 4-qubit GHZ state

Extend the GHZ circuit to 4 qubits.

1. Build the circuit and verify with `Statevector.probabilities_dict()` that only `'0000'` and `'1111'` appear.
2. Measure 4096 shots and confirm.
3. What happens if you measure qubit 3 and discard it, then measure qubits 0, 1, 2? Are they still correlated?

In [ ]:
# Exercise 2: your solution here
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector

# TODO: build 4-qubit GHZ circuit
# TODO: verify probabilities
# TODO: partial measurement

### Exercise 3: W state verification

Verify the W state has the correct structure.

1. Use `Statevector` to confirm that only `'001'`, `'010'`, and `'100'` have non-zero amplitude, each equal to $1/\sqrt{3}$.
2. Run 6000 shots and verify each outcome appears approximately 2000 times.
3. Measure qubit 2 and discard it. Show that when qubit 2 measured 0, qubits 0 and 1 show outcomes `'01'` and `'10'` with equal probability (entangled), and when qubit 2 measured 1, qubits 0 and 1 are both 0 (unentangled).

In [ ]:
# Exercise 3: your solution here
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector

qc_w = make_W_state()

# 1. Statevector verification
# TODO

# 2. Shot-based measurement
# TODO

# 3. Partial measurement and entanglement check
# TODO

### Exercise 4: CHSH with |Ψ-⟩

Repeat the CHSH measurement using $|\Psi^-\rangle$ instead of $|\Phi^+\rangle$.

For $|\Psi^-\rangle$, the correlation function is $E(\alpha,\beta) = -\cos(2(\alpha-\beta))$.

1. Modify `measure_correlation` to prepare $|\Psi^-\rangle$ (using `make_bell(x0=True, x1=True)`).
2. Use the same optimal angles. Do you still get a violation?
3. What value of S do you expect? Compute it analytically and verify against the simulation.

In [ ]:
# Exercise 4: your solution here
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

sim = AerSimulator()

def measure_correlation_psi_minus(angle_a, angle_b, shots=8192):
    """CHSH correlation for |Ψ-⟩."""
    qc = QuantumCircuit(2, 2)
    # TODO: prepare |Ψ-⟩
    qc.ry(2 * angle_a, 0)
    qc.ry(2 * angle_b, 1)
    qc.measure([0, 1], [0, 1])
    counts = sim.run(transpile(qc, sim), shots=shots).result().get_counts()
    n00 = counts.get('00', 0)
    n01 = counts.get('01', 0)
    n10 = counts.get('10', 0)
    n11 = counts.get('11', 0)
    return (n00 + n11 - n01 - n10) / shots

# TODO: compute S and compare to classical bound and theoretical prediction

## Summary

In this lesson you:

- Verified that H followed by CNOT produces an entangled state whose statevector cannot be factored.
- Prepared all four Bell states using `make_bell` and verified with `Statevector`.
- Used the inverse Bell circuit (CNOT then H) to uniquely decode each Bell state.
- Observed perfect correlations in $|\Phi^+\rangle$ measurements and contrasted with the separable case.
- Built the GHZ state with a two-CNOT chain and the W state using $R_y$, `ch`, and Toffoli.
- Demonstrated that GHZ loses all entanglement under qubit loss, while W retains entanglement in the remaining pair.
- Measured the CHSH parameter $S \approx 2.83$ for $|\Phi^+\rangle$, exceeding the classical bound of 2.

**Lesson 6** applies Bell states to quantum communication: teleportation transfers an unknown qubit state using classical bits and shared entanglement, and superdense coding doubles the classical capacity of one qubit channel.